# Task 5 – Confronto tra ordinamento naturale e nested dissection

Risolviamo il sistema per $N = 32, 64, \dots, 1024$ con i due ordinamenti e confrontiamo:

- il **numero di non zeri** del fattore $L$ (memoria richiesta);
- i **tempi** di fattorizzazione e di soluzione;
- la **struttura** delle matrici $A$ e $L$, con `matplotlib.pyplot.spy`;
- la **soluzione** $u$ per diversi valori di $N$.

Le funzioni sono le stesse del notebook `task4.ipynb`, nella versione corretta; le ripetiamo qui perché questo notebook si possa eseguire da solo. **Attenzione:** se si modifica una funzione in uno dei due notebook, va modificata anche nell'altro.

In [ ]:
import json          # per salvare i risultati su file dopo ogni esperimento
import subprocess   # per lanciare task1, task2, task3 dal notebook
import time         # per misurare i tempi
from importlib.metadata import version

import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix, csc_matrix
from scipy.sparse.linalg import spsolve_triangular
from sksparse.cholmod import cholesky

print("scikit-sparse", version("scikit-sparse"))

## Funzioni

Sono quelle del Task 4, con le stesse correzioni:

- `esegui` ferma il notebook **mostrando il messaggio di errore** del programma C++, se ne fallisce uno;
- `risolvi` cancella la copia CSR di $L$ appena risolto il primo sistema, e per il secondo usa `L.T`, che è già in CSR senza copie: con $N$ grande ogni copia di $L$ occupa GB.

In più c'è `misura`, che ripete la risoluzione e tiene il tempo **minimo**. Con $N$ piccoli una misura dura pochi millisecondi e risente di costi fissi e di rumore (altri processi, cache); il minimo di alcune ripetizioni è la stima meno disturbata. Con $N$ grandi i tempi sono di secondi e una misura sola basta, quindi ripetiamo solo fino a `N_RIPETI`.

In [ ]:
def esegui(cmd):
    # Lancia un programma; se termina con errore, mostra il suo messaggio su cerr.
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{cmd[0]} fallito (codice {r.returncode}):\n{r.stderr}")


def genera(N, scelta):
    # Catena dei programmi C++: task1 N -> task2 -> task3 scelta
    esegui(["./task1", str(N)])
    esegui(["./task2"])
    esegui(["./task3", scelta])


def leggi_sistema():
    # A.txt in formato COO (triple r c valore), convertito in CSC
    dati = np.fromfile("A.txt", sep=" ").reshape(-1, 3)
    b = np.fromfile("rhs.txt", sep=" ")
    n = len(b)
    A = coo_matrix((dati[:, 2], (dati[:, 0].astype(int), dati[:, 1].astype(int))),
                   shape=(n, n)).tocsc()
    return A, b


def my_cholesky(A):
    # scikit-sparse 0.5.0: cholesky restituisce (R, p) con A = R^T R,
    # R triangolare superiore; il fattore inferiore e' L = R^T
    # (verificato nel notebook task4.ipynb).
    R, p = cholesky(A, order="natural")
    assert np.array_equal(p, np.arange(A.shape[0])), "CHOLMOD ha riordinato la matrice"
    return csc_matrix(R.T)


def risolvi(A, b):
    t0 = time.perf_counter()
    L = my_cholesky(-A)                  # L in formato CSC
    t1 = time.perf_counter()

    L_csr = L.tocsr()                    # copia di L in CSR, per Ly = -b
    y = spsolve_triangular(L_csr, -b, lower=True)
    del L_csr                            # liberiamo subito la copia

    # L.T e' gia' CSR (trasposta di una CSC): nessuna copia
    u = spsolve_triangular(L.T, y, lower=False)
    t2 = time.perf_counter()

    residuo = np.linalg.norm(A @ u - b) / np.linalg.norm(b)
    return u, L, t1 - t0, t2 - t1, residuo


def misura(A, b, ripetizioni):
    # Ripete la risoluzione e tiene il tempo minimo di fattorizzazione e di
    # soluzione. La soluzione e L sono quelle dell'ultima ripetizione
    # (sono sempre le stesse: cambia solo il tempo impiegato).
    t_fatt, t_sol = float("inf"), float("inf")
    for _ in range(ripetizioni):
        u, L, tf, ts, res = risolvi(A, b)
        t_fatt, t_sol = min(t_fatt, tf), min(t_sol, ts)
    return u, L, t_fatt, t_sol, res


def in_ordine_naturale(u, scelta):
    if scelta == "naturale":
        return u
    ordine = np.fromfile("ordering.txt", sep=" ").reshape(-1, 2).astype(int)
    u_nat = np.empty_like(u)
    u_nat[ordine[:, 1]] = u[ordine[:, 0]]
    return u_nat

## 1. Cosa ci aspettiamo dalla teoria

**Ordinamento naturale.** Con la numerazione per righe, il nodo $n$ è collegato a $n \pm 1$ e $n \pm N$: $A$ è una matrice a **banda** di semiampiezza $N$. La fattorizzazione di Cholesky conserva la banda ma la **riempie** tutta: ogni riga di $L$ ha circa $N$ non zeri, quindi

$$\text{nnz}(L) \approx N^2 \cdot N = N^3, \qquad \text{costo della fattorizzazione} \approx N^2 \cdot N^2 = N^4.$$

**Nested dissection.** Il separatore al livello più alto ha $N$ nodi e, messo in fondo, produce un blocco pieno di circa $N^2/2$ entrate; i livelli sotto lavorano su sottoproblemi sempre più piccoli. Sommando su tutti i livelli si ottiene (risultato di A. George, 1973)

$$\text{nnz}(L) = O(N^2 \log N), \qquad \text{costo della fattorizzazione} = O(N^3).$$

In un grafico log-log una legge $N^k$ è una retta di pendenza $k$: per $\text{nnz}(L)$ ci aspettiamo pendenza circa 3 per il naturale e poco più di 2 per il nested dissection; per il tempo di fattorizzazione circa 4 e 3.

**Limite di memoria.** Con l'ordinamento naturale e $N = 1024$ si avrebbe $\text{nnz}(L) \approx 1024^3 \approx 10^9$: servirebbero circa 12 GB (8 byte per il valore e 4 per l'indice di riga di ogni entrata), più della memoria di un Codespace standard. Il calcolo verrebbe interrotto dal sistema operativo e il kernel del notebook si riavvierebbe. Per questo `N_MAX_NATURALE` limita l'ordinamento naturale: se la macchina ha abbastanza memoria, si può alzare a 1024.

## 2. Organizzazione degli esperimenti

**Ordine.** Eseguiamo prima **tutti** i casi con il nested dissection, che richiedono poca memoria, e solo dopo quelli con l'ordinamento naturale, partendo dai più piccoli. Così il caso più pesante (naturale con `N_MAX_NATURALE`) è l'ultimo: se esaurisse la memoria, tutti gli altri risultati sarebbero già calcolati.

**Salvataggio.** Dopo ogni esperimento i risultati vengono scritti in `risultati_task5.json`, e le soluzioni del nested dissection in `soluzioni_task5.npz`. Se il kernel si riavvia per mancanza di memoria, non si perde nulla: basta eseguire la cella di ripresa della sezione 5 e proseguire con i grafici, eventualmente abbassando `N_MAX_NATURALE`.

**Memoria con $N = 512$ naturale.** $\text{nnz}(L) \approx 512^3 \approx 1.3 \cdot 10^8$, circa 1.6 GB per ogni copia di $L$; con le copie temporanee il picco può arrivare a qualche GB. Su un Codespace base (8 GB) è al limite: se non riesce, si imposta `N_MAX_NATURALE = 256` e lo si commenta nelle conclusioni.

In [ ]:
VALORI_N = [32, 64, 128, 256, 512, 1024]
N_MAX_NATURALE = 512        # vedi la nota sulla memoria qui sopra
N_RIPETI = 128              # fino a questo N si ripete la misura
RIPETIZIONI = 3

FILE_RISULTATI = "risultati_task5.json"
FILE_SOLUZIONI = "soluzioni_task5.npz"

risultati = {"naturale": {}, "riordinato": {}}
soluzioni = {}               # soluzioni del nested dissection in ordine naturale


def salva_risultati():
    # json vuole chiavi stringa: N diventa "32", "64", ... (vedi la ripresa)
    with open(FILE_RISULTATI, "w") as fh:
        json.dump(risultati, fh, indent=1)


def esperimento(N, scelta):
    genera(N, scelta)
    A, b = leggi_sistema()
    rip = RIPETIZIONI if N <= N_RIPETI else 1
    u, L, t_fatt, t_sol, res = misura(A, b, rip)

    # int() e float(): i tipi di NumPy non sempre si salvano in json
    risultati[scelta][N] = {"nnzA": int(A.nnz), "nnzL": int(L.nnz),
                            "t_fatt": float(t_fatt), "t_sol": float(t_sol),
                            "res": float(res)}
    salva_risultati()
    print(f"N = {N:5d} {scelta:>10}: nnz(L) = {L.nnz:>11,}  "
          f"fatt. {t_fatt:8.3f} s  sol. {t_sol:8.3f} s  residuo {res:.1e}")
    return in_ordine_naturale(u, scelta)

## 3. Esperimenti con il nested dissection

Tutti i valori di $N$, fino a 1024.

In [ ]:
for N in VALORI_N:
    soluzioni[N] = esperimento(N, "riordinato")

# Le soluzioni servono per i grafici della sezione 9: le salviamo su file
np.savez(FILE_SOLUZIONI, **{str(N): u for N, u in soluzioni.items()})

## 4. Esperimenti con l'ordinamento naturale

Fino a `N_MAX_NATURALE`, dal più piccolo al più grande. I risultati vengono salvati dopo ogni caso.

In [ ]:
for N in VALORI_N:
    if N > N_MAX_NATURALE:
        break
    esperimento(N, "naturale")   # la soluzione non serve: e' uguale a quella riordinata

## 5. Ripresa dopo un riavvio del kernel (solo se serve)

Se il kernel si è riavviato durante la sezione 4, i risultati in memoria sono persi ma quelli su file no. In quel caso: eseguire la cella degli import, quella delle funzioni e quella dei parametri (sezione 2), poi **questa** cella, e proseguire dalla sezione 6. In un'esecuzione normale questa cella si può saltare: ricarica gli stessi dati che sono già in memoria.

In [ ]:
with open(FILE_RISULTATI) as fh:
    dati = json.load(fh)
# Le chiavi tornano da stringhe a interi: "32" -> 32
risultati = {scelta: {int(N): v for N, v in dati[scelta].items()} for scelta in dati}
soluzioni = {int(N): u for N, u in np.load(FILE_SOLUZIONI).items()}
print("Naturale:  ", sorted(risultati["naturale"]))
print("Riordinato:", sorted(risultati["riordinato"]))

## 6. Tabella riassuntiva

Il **rapporto** è $\text{nnz}(L)$ naturale diviso $\text{nnz}(L)$ riordinato: dice quante volte più memoria richiede l'ordinamento naturale.

In [ ]:
print(f"{'N':>5} | {'nnz(A)':>10} | {'nnz(L) naturale':>16} | {'nnz(L) riordinato':>17} | {'rapporto':>8}")
print("-" * 70)
for N in VALORI_N:
    nat = risultati["naturale"].get(N)
    rio = risultati["riordinato"][N]
    if nat:
        print(f"{N:5d} | {rio['nnzA']:10,} | {nat['nnzL']:16,} | {rio['nnzL']:17,} | "
              f"{nat['nnzL'] / rio['nnzL']:8.1f}")
    else:
        print(f"{N:5d} | {rio['nnzA']:10,} | {'(non calcolato)':>16} | {rio['nnzL']:17,} | {'-':>8}")

## 7. Non zeri del fattore L

Grafico log-log con le rette di riferimento $N^3$ e $N^2 \log_2 N$, scalate per passare per il primo punto di ciascuna serie: se i dati seguono la retta, la crescita è quella prevista dalla teoria.

La retta $N^3$ prosegue anche oltre `N_MAX_NATURALE`: indica quanti non zeri avrebbe il fattore naturale nei casi che non è stato possibile calcolare.

In [ ]:
def serie(scelta, chiave):
    Ns = sorted(risultati[scelta])
    return np.array(Ns), np.array([risultati[scelta][N][chiave] for N in Ns])

plt.figure(figsize=(6, 4.5))
Nn, Ln = serie("naturale", "nnzL")
Nr, Lr = serie("riordinato", "nnzL")
plt.loglog(Nn, Ln, "o-", label="naturale")
plt.loglog(Nr, Lr, "s-", label="nested dissection")

Nrif = np.array(VALORI_N, dtype=float)
plt.loglog(Nrif, Ln[0] * (Nrif / Nn[0])**3, "k--", lw=1, label=r"$\propto N^3$")
plt.loglog(Nrif, Lr[0] * (Nrif**2 * np.log2(Nrif)) / (Nr[0]**2 * np.log2(Nr[0])),
           "k:", lw=1, label=r"$\propto N^2 \log N$")

plt.xlabel("N")
plt.ylabel("nnz(L)")
plt.title("Non zeri del fattore di Cholesky")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

# Pendenza stimata tra gli ultimi due punti di ogni serie
for nome, Ns, Ls in [("naturale", Nn, Ln), ("riordinato", Nr, Lr)]:
    k = np.log(Ls[-1] / Ls[-2]) / np.log(Ns[-1] / Ns[-2])
    print(f"{nome}: pendenza stimata {k:.2f}")

## 8. Tempi di fattorizzazione e di soluzione

Il tempo di fattorizzazione dipende dal numero di operazioni (teoria: $N^4$ contro $N^3$); quello di soluzione è proporzionale a $\text{nnz}(L)$, perché le due sostituzioni toccano ogni entrata di $L$ una volta. Per $N$ piccoli i tempi sono così brevi che pesano i costi fissi (chiamate di libreria, conversioni di formato), quindi le pendenze vanno lette sui valori grandi.

In [ ]:
fig, assi = plt.subplots(1, 2, figsize=(11, 4.5))
for asse, chiave, titolo in [(assi[0], "t_fatt", "Fattorizzazione"),
                             (assi[1], "t_sol", "Soluzione (due sistemi triangolari)")]:
    for scelta, stile, nome in [("naturale", "o-", "naturale"),
                                ("riordinato", "s-", "nested dissection")]:
        Ns, ts = serie(scelta, chiave)
        asse.loglog(Ns, ts, stile, label=nome)
    asse.set_xlabel("N")
    asse.set_ylabel("tempo (s)")
    asse.set_title(titolo)
    asse.legend()
    asse.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

# Pendenza stimata del tempo di fattorizzazione tra gli ultimi due punti
for scelta in ["naturale", "riordinato"]:
    Ns, ts = serie(scelta, "t_fatt")
    k = np.log(ts[-1] / ts[-2]) / np.log(Ns[-1] / Ns[-2])
    print(f"{scelta}: pendenza stimata del tempo di fattorizzazione {k:.2f}")

## 9. Struttura delle matrici (spy)

`spy` disegna un punto per ogni entrata non nulla. Con $N = 32$ ($1024 \times 1024$):

- **$A$ naturale**: cinque diagonali, la principale, le due a distanza 1 e le due a distanza $N$;
- **$L$ naturale**: tutta la banda tra la diagonale e la distanza $N$ viene riempita (*fill-in*);
- **$A$ riordinata**: una struttura a blocchi; i blocchi tra $V_1$ e $V_2$ sono vuoti, e le righe e colonne in fondo sono i separatori;
- **$L$ riordinata**: i blocchi vuoti restano vuoti, il riempimento si concentra nei separatori, soprattutto nell'angolo in basso a destra.

Le matrici si ricalcolano qui per $N = 32$ (pochi millisecondi): così questa sezione funziona anche dopo una ripresa.

In [ ]:
matrici = {}
for scelta in ["naturale", "riordinato"]:
    genera(32, scelta)
    A, b = leggi_sistema()
    matrici[scelta] = (A, my_cholesky(-A))

fig, assi = plt.subplots(2, 2, figsize=(10, 10))
for riga, scelta in enumerate(["naturale", "riordinato"]):
    A, L = matrici[scelta]
    assi[riga, 0].spy(A, markersize=0.3)
    assi[riga, 0].set_title(f"A, {scelta} (nnz = {A.nnz:,})")
    assi[riga, 1].spy(L, markersize=0.3)
    assi[riga, 1].set_title(f"L, {scelta} (nnz = {L.nnz:,})")
plt.tight_layout()
plt.show()

## 10. Soluzione per diversi valori di N

Aumentando $N$ la griglia si infittisce e la soluzione discreta approssima sempre meglio quella continua: l'aspetto non cambia, ma diventa più liscio. Il valore massimo si stabilizza, segno che la discretizzazione converge.

In [ ]:
mostra = [32, 128, 512]
fig, assi = plt.subplots(1, len(mostra), figsize=(15, 4.5))
for asse, N in zip(assi, mostra):
    h = 1 / (N + 1)
    U = soluzioni[N].reshape(N, N)
    im = asse.imshow(U, origin="lower", extent=[h, 1 - h, h, 1 - h], cmap="inferno")
    asse.set_title(f"N = {N}, max u = {U.max():.4f}")
    asse.set_xlabel("x")
    asse.set_ylabel("y")
    fig.colorbar(im, ax=asse, shrink=0.8)
plt.tight_layout()
plt.show()

## 11. Conclusioni

*Da completare dopo aver eseguito il notebook, con i propri numeri.* Punti da commentare:

- di quanto si riduce $\text{nnz}(L)$ con il riordinamento, e come cresce il rapporto con $N$;
- se le pendenze stimate sono vicine a 3 (naturale) e a circa 2 (nested dissection);
- il guadagno nei tempi di fattorizzazione e di soluzione;
- perché l'ordinamento naturale non si può eseguire per $N = 1024$ con la memoria disponibile, mentre il nested dissection sì;
- che il residuo resta dell'ordine di $10^{-14}$ in tutti i casi: il riordinamento cambia il costo, non la precisione.